In [2]:
import torch
import torch.nn.functional as F
import torch.nn as nn
# import torch.optim as optim
# import torch_optimizer as optim
import torch.nn.init as init
import adabound
import matplotlib.pyplot as plt
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import math
from sklearn.model_selection import train_test_split

from tqdm import tqdm
from tqdm import trange

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/zorocrit/Control_Nuclear_Spins/master/C13spin/test_data/atesteddata_20230322_102258.csv?token=GHSAT0AAAAAAB5HMDTBSOWTNMWWSHFD2MQAZA2SSKQ')
# df

In [4]:
# y = df[['x', 'N', 'z']]
# # y

y = df[['Xtau', 'XN', 'Ztau', 'ZN']]
y

,Xtau,XN,Ztau,ZN
0,2.959641,10.0,0.176380,1.0
1,4.245750,7.0,0.625249,22.0
2,3.594517,10.0,1.129403,1.0
3,0.479873,16.0,0.059788,1.0
4,2.861716,13.0,0.197018,4.0
...,...,...,...,...
1312,2.671803,10.0,0.143248,7.0
1313,2.504895,7.0,1.044665,1.0
1314,3.934559,16.0,0.737077,1.0
1315,3.627678,10.0,0.214985,1.0


In [5]:
X = df[['Al', 'Ap']]
# X

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.10, random_state=100)

In [14]:
Xdata = list(np.array(X_train.values.tolist()))
print(Xdata.__len__())

1185


In [8]:
ydata = list(np.array(y_train.values.tolist()))
# print(ydata)

In [9]:
torch.manual_seed(1)

In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# for reproducibility
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

In [11]:
x = torch.FloatTensor(Xdata).to(device)
y = torch.FloatTensor(ydata).to(device)

c:\Users\Administrator\anaconda3\lib\site-packages\ipykernel_launcher.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at  C:\b\abs_bao0hdcrdh\croot\pytorch_1675190257512\work\torch\csrc\utils\tensor_new.cpp:204.)
  """Entry point for launching an IPython kernel.


In [16]:
num_data = 1185

num_epoch = 10000

In [19]:
from sys import stdout


model = nn.Sequential(

    nn.Linear(2,400),

    nn.ReLU(),

    nn.Linear(400,400),

    nn.ReLU(),

    nn.Linear(400,400),

    nn.ReLU(),
    
    # nn.Linear(400,400),

    # nn.ReLU(),
    
    # nn.Linear(400,400),

    # nn.ReLU(),
        
    # nn.Linear(400,400),

    # nn.ReLU(),
    
    nn.Linear(400,400),

    nn.ReLU(),

    nn.Linear(400,4)

).to(device)

#ReLU라는 Activation Function을 사용하여, 4개의 Linear Layer로 모델 구현

# Input layer에 1개씩 데이터가 들어가므로 nn.Linear(1,2)이며, 최종적으로 1개의 값이 나와야하기에 Output Layer는 nn.Linear(4,1)

 

loss_func = nn.L1Loss().to(device)

# optimizer = optim.SGD(model.parameters(),lr = 0.0002)

# optimizer = adabound.AdaBound(model.parameters(), lr=0.935 * 1e-4, final_lr=0.05758)  #Loss: 0.0537

# optimizer = adabound.AdaBound(model.parameters(), lr=0.435 * 1e-3, final_lr=0.05558)

optimizer = adabound.AdaBound(model.parameters(), lr=0.935 * 1e-4, final_lr=0.04958)  #Loss: 0.0537

loss_array = []

minloss = 10

for i in tqdm(range(num_epoch)) : 

    optimizer.zero_grad()

    output = model(x)

    loss = loss_func(output,y)

    loss.backward()

    optimizer.step()

    if(loss < minloss):
       minloss = loss

    loss_array.append(loss)
    if(i == 1):
      stdout.write("Minimum loss: " + str(minloss))
    if(i%1000 == 0 and i != 0):
      # print("Case "+ str(i) + ", Loss: " + str(loss.data))
      stdout.write("Minimum loss: " + str(minloss))

    if i == num_epoch - 1:

        print(loss.data)

        param_list = list(model.parameters())

        #최종 학습된 마지막 결과물의 Parameter 저장

        print(param_list)

loss_array = torch.tensor(loss_array)

loss_array.detach().numpy()

plt.plot(loss_array)

plt.show()

#Loss(y_predicted - y_real)값이 어떻게 변하는지 그래프로 도식화

Traceback (most recent call last):
  File "_pydevd_bundle/pydevd_cython.pyx", line 1078, in _pydevd_bundle.pydevd_cython.PyDBFrame.trace_dispatch
  File "_pydevd_bundle/pydevd_cython.pyx", line 297, in _pydevd_bundle.pydevd_cython.PyDBFrame.do_wait_suspend
  File "c:\Users\Administrator\anaconda3\lib\site-packages\debugpy\_vendored\pydevd\pydevd.py", line 1976, in do_wait_suspend
    keep_suspended = self._do_wait_suspend(thread, frame, event, arg, suspend_type, from_this_thread, frames_tracker)
  File "c:\Users\Administrator\anaconda3\lib\site-packages\debugpy\_vendored\pydevd\pydevd.py", line 2011, in _do_wait_suspend
    time.sleep(0.01)
KeyboardInterrupt


KeyboardInterrupt: 

In [75]:
# # 임의의 입력 [73, 80, 75]를 선언
# new_var =  torch.FloatTensor([[4.493041, 0.5427]]).to(device)
# # 입력한 값 [73, 80, 75]에 대해서 예측값 y를 리턴받아서 pred_y에 저장
# pred_y = model(new_var) 
# print("훈련 후 예측값 :", pred_y) 

훈련 후 예측값 : tensor([[ 3.4226, 16.2565,  2.0645, 12.0586]], device='cuda:0',
       grad_fn=<AddmmBackward0>)


In [ ]:
model._save_to_state_dict("model_20230322.h5")

In [19]:
model = nn.Sequential(

    nn.Linear(2,400),

    nn.ReLU(),

    nn.Linear(400,400),

    nn.ReLU(),

    nn.Linear(400,400),

    nn.ReLU(),
    
    nn.Linear(400,400),

    nn.ReLU(),
    
    # nn.Linear(400,400),

    # nn.ReLU(),
        
    # nn.Linear(400,400),

    # nn.ReLU(),
    
    # nn.Linear(400,400),

    # nn.ReLU(),

    nn.Linear(400,4)

).to(device)

model.load_state_dict("CNN_Model_20230307_144034.h5")

TypeError: Expected state_dict to be dict-like, got <class 'str'>.

In [1]:
y_input = model(X_valid.to(device))
y_input

NameError: name 'model' is not defined